In [2]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from ITIS_Model_funcs import get_nominal_param,OLS_res,ITIS

In [3]:
param_log,IC = get_nominal_param()

pickles = ['11', '12', '21', '22', '31', '32']

with open('OLS_Results\\ResAnalysis11.pkl', 'rb') as f:
    results = pickle.load(f)

        ##Reference
        # all_results = {
        #     'output_ids': output_ids,
        #     'param_ids' : param_ids,
        #     'param_in' : param_in,
        #     'twonorms' : twonorms,
        #     't_data' : t_data,
        #     'y_data' : y_data,
        #     'opt_model' : opt_model,
        #     'true_sol' : true_sol,
        #     'param_opt' : param_opt,
        #     'optimal_solution' :np.exp(least_sq_sol.x),
        #     'objvalue' : least_sq_sol.cost
        # }


output_ids = results['output_ids']
t_data = results['t_data']
param_opt = results['param_opt']
y_data = results['y_data']
param_ids = results['param_ids']

##Sensitvity analysis of residuals Starting from optimized param values?

##SENSITIVITY ANALYSIS
h = 1e-6  #amount to perturb parameters
n_param = len(param_opt)
n_states = len(output_ids)

Sensitivity_Mat = np.zeros((n_param, len(t_data) * n_states))  ##Initialize shape of sensitivity matrix.


##Reference
# for i in range(n_param):  #calculate the relative sensitivity to each 45 parameters
#
#         param_in = param_log[i]
#         param_delta = param_in + h ##    [HOW EXACTLY SHOULD I BE PERTURBING MY PARAM?]
#         ##THIS THING NEXT DO THIS!!!
#         Sensitivity_Mat[i, :] = ((1 / h) * (OLS_res(param_delta, y_data, t_data, i, output_ids, param_all, IC)
#                                             - OLS_res(param_in, y_data, t_data, i, output_ids, param_all, IC)))
for i in range(n_param):  #calculate the relative sensitivity to each 45 parameters

        param_in = param_opt[i]
        param_delta = param_in + h ##    [HOW EXACTLY SHOULD I BE PERTURBING MY PARAM?]
        ##THIS THING NEXT DO THIS!!!
        Sensitivity_Mat[i, :] = ((1 / h) * (OLS_res(param_delta, y_data, t_data, i, output_ids, param_log, IC)
                                            - OLS_res(param_in, y_data, t_data, i, output_ids, param_log, IC)))


print(Sensitivity_Mat)

[[ 0.          0.          0.02569191 ...  1.23501081 -4.6882761
  -0.72166244]
 [ 0.          0.         -0.23590286 ...  0.19435676 -0.08337292
  -0.40518055]
 [ 0.          0.         -0.0078144  ... -0.29959346  1.4531884
   0.62260795]
 ...
 [-1.57719212 -0.66277466 -0.07660752 ...  0.61149354 -2.56526676
  -1.08932645]
 [ 1.28723068  1.13002823 -0.32107369 ... -0.57961309  0.48021632
   0.4601156 ]
 [-1.96115535 -0.94313982 -0.07007437 ...  0.22227814 -2.012977
  -0.76833596]]


In [10]:
F = Sensitivity_Mat@Sensitivity_Mat.T
print(np.shape(F))
np.linalg.cond(F)

(45, 45)


np.float64(9867478140700.469)

In [18]:
S_opt = Sensitivity_Mat[param_ids,:]
F_opt = S_opt@S_opt.T
np.linalg.cond(F_opt)
C = np.linalg.inv(F_opt)
print(np.diag(C))
print(np.exp(param_opt[param_ids]))
print(np.diag(C)/np.exp(param_opt[param_ids]))


[0.00240548 0.00241232 0.00521131 0.09304717 0.01302659 0.04705096
 0.07391414 0.0236107  0.32965944 0.00245302]
[1.51808303e-02 2.01028096e-01 3.04510970e-02 8.74918731e+01
 2.95522722e-09 5.00624103e-04 8.43646030e+01 1.89949090e+03
 1.14852604e-01 8.50500412e+02]
[1.58455133e-01 1.19999335e-02 1.71137028e-01 1.06349502e-03
 4.40798332e+06 9.39846135e+01 8.76127454e-04 1.24300167e-05
 2.87028268e+00 2.88420651e-06]
